<a href="https://colab.research.google.com/github/OdysseusPolymetis/initiation_ia/blob/main/StableDiffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stable Diffusion : générer une image à partir d'un texte

Dans ce notebook, on change de logique.

Jusqu'ici, nous avons vu :
- des modèles qui génèrent à partir d'un latent ;
- des modèles qui traduisent une image d'un domaine à un autre ;
- des modèles qui modifient une image en s'appuyant sur un texte.

Ici, nous allons utiliser un modèle qui génère directement une image à partir d'un **prompt textuel**.

## Objectif du notebook
Nous allons :
1. charger une pipeline Stable Diffusion ;
2. écrire un prompt ;
3. générer une image ;
4. comparer plusieurs prompts et plusieurs réglages.

## Idée générale
Le texte sert ici de guide principal pour la génération.

## Où se situe Stable Diffusion dans le cours ?

Jusqu'ici :

- **StyleGAN2** : générer une image à partir d'un latent ;
- **StyleGAN3** : interpoler et animer ;
- **StyleCLIP** : guider une édition avec du texte ;
- **CycleGAN** : traduire une image d'un domaine à un autre.

Maintenant :

- **Stable Diffusion** : générer directement une image à partir d'une description textuelle.

## Idée clé
Le texte ne sert plus seulement à guider une modification.
Il sert à piloter directement la génération.

## Qu'est-ce qu'un modèle de diffusion ?

L'idée générale est la suivante :

1. on part d'un bruit aléatoire ;
2. le modèle enlève progressivement ce bruit ;
3. le texte guide ce processus ;
4. à la fin, on obtient une image.

## Important
L'image n'est pas produite d'un seul coup.
Elle apparaît progressivement au cours du débruitage.

## Pourquoi parle-t-on de "latent diffusion" ?

Stable Diffusion ne travaille pas directement sur tous les pixels de l'image finale.

Il travaille d'abord dans un **espace latent**, c'est-à-dire une représentation compressée de l'image.

## Pourquoi est-ce utile ?
Cela permet de réduire le coût de calcul
et de rendre la génération plus accessible.

In [ ]:
!nvidia-smi || true

import sys
import platform
import torch

print("Python :", sys.version)
print("Plateforme :", platform.platform())
print("Torch version :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

## Installer les bibliothèques utiles

Nous allons utiliser la bibliothèque **Diffusers** de Hugging Face.

Elle fournit des pipelines prêtes à l'emploi pour les modèles de diffusion,
dont Stable Diffusion.

In [ ]:
!pip -q install diffusers transformers accelerate safetensors

## Charger la pipeline Stable Diffusion

Nous allons charger un modèle Stable Diffusion 2.1.

## Important
Le téléchargement peut prendre un peu de temps la première fois.

## Remarque
Selon le runtime Colab, il peut être nécessaire d'être connecté à Hugging Face
pour certains modèles. Si besoin, on pourra adapter ensuite.

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    use_safetensors=True
)

pipe = pipe.to("cuda" if torch.cuda.is_available() else "cpu")
print("Pipeline chargée.")

## Choisir un prompt

Le **prompt** est la description textuelle de l'image que l'on veut générer.

## Conseils pour débuter
Un prompt simple fonctionne souvent mieux pour une première démonstration.

Exemple :
- "a realistic portrait of a woman"
- "a winter mountain landscape"
- "a black cat sitting on a chair"

In [ ]:
prompt = "a winter mountain landscape, realistic photo"
negative_prompt = "blurry, low quality, distorted"

print("Prompt positif :", prompt)
print("Prompt négatif :", negative_prompt)

## Comprendre le seed dans Stable Diffusion

Le seed joue encore un rôle important.

### À quoi sert-il ?
Il fixe le bruit de départ.

### Conséquence
- même prompt + même seed + mêmes paramètres = image identique ;
- même prompt + seed différent = image différente.

Le seed ne décrit pas l'image.
Il fixe simplement le point de départ du bruit aléatoire.

In [ ]:
seed = 42
generator = torch.Generator(device=pipe.device).manual_seed(seed)

print("Seed utilisé :", seed)

## Première génération

Nous allons maintenant générer une image.

### Paramètres importants
- `num_inference_steps` : nombre d'étapes de débruitage ;
- `guidance_scale` : force avec laquelle le texte guide la génération.

### Intuition
- plus de steps : plus de calcul, souvent plus de détail ;
- guidance plus élevée : le modèle suit davantage le texte.

In [ ]:
image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=generator
).images[0]

image

## Générer plusieurs images avec le même prompt mais des seeds différents

On garde le même texte,
mais on change le seed.

## Pourquoi ?
Cela permet de montrer que le prompt ne détermine pas une seule image,
mais une famille d'images compatibles avec la description.

In [ ]:
seeds = [1, 2, 3, 4]

images = []
for s in seeds:
    gen = torch.Generator(device=pipe.device).manual_seed(s)
    img = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=gen
    ).images[0]
    images.append((s, img))

In [ ]:
plt.figure(figsize=(12,12))

for i, (s, img) in enumerate(images, start=1):
    plt.subplot(2, 2, i)
    plt.imshow(img)
    plt.title(f"seed = {s}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## Que montre cette comparaison ?

Le prompt fixe une direction générale,
mais il laisse de nombreuses possibilités de réalisation.

## Idée clé
Le modèle ne produit pas une seule image "correcte".
Il explore plusieurs solutions compatibles avec le texte.

## Comprendre le guidance scale

Le **guidance scale** contrôle à quel point le modèle suit le texte.

### Intuition simple
- guidance plus faible : plus de liberté ;
- guidance plus forte : le texte impose davantage sa volonté.

### Attention
Une guidance trop forte ne donne pas toujours de meilleurs résultats.

In [ ]:
guidance_values = [3.0, 7.5, 12.0]
guidance_images = []

for g in guidance_values:
    gen = torch.Generator(device=pipe.device).manual_seed(42)
    img = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=30,
        guidance_scale=g,
        generator=gen
    ).images[0]
    guidance_images.append((g, img))

In [ ]:
plt.figure(figsize=(15,5))

for i, (g, img) in enumerate(guidance_images, start=1):
    plt.subplot(1, 3, i)
    plt.imshow(img)
    plt.title(f"guidance = {g}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## Que faut-il observer ?

On garde ici :
- le même prompt ;
- le même seed ;
- le même nombre de steps.

La seule chose qui change est le guidance scale.

### Questions
- quelle image suit le mieux le prompt ?
- quelle image semble la plus naturelle ?
- à partir de quand le guidage paraît-il trop fort ?

## Ce qu'il faut retenir

Dans ce notebook, on a vu que :

- Stable Diffusion est un modèle text-to-image ;
- il part d'un bruit latent ;
- le texte guide le processus de débruitage ;
- le seed fixe le bruit de départ ;
- le guidance scale règle la force du guidage textuel.

## Formule simple

**texte + bruit initial -> débruitage progressif -> image**